In [1]:
import requests
from bs4 import BeautifulSoup

# Data Collection
The Swissgrid website does not contain constant URL's, only the last heading for the different segments remains constant. Thus the URL's need to be scraped before any data wrangling can be done.

In [2]:
# import re
import requests
from bs4 import BeautifulSoup
# from urllib.parse import urljoin

PAGE_URL = "https://www.swissgrid.ch/en/home/customers/topics/energy-data-ch.html"
FILE_DAILY = "https://www.swissgrid.ch/dam/jcr:0bbafc76-e5bc-4126-bd99-41c939fe10e0/control-area-balance-daily.csv"
FILE_2026 = "https://www.swissgrid.ch/dam/jcr:e329e825-6d06-4713-9551-54e301f32095/control-area-balance-2026.xlsx"

In [ ]:
# import re
import requests
from bs4 import BeautifulSoup
# from urllib.parse import urljoin

PAGE_URL = "https://www.swissgrid.ch/en/home/customers/topics/energy-data-ch.html"

response = requests.get(PAGE_URL, timeout=30)
response.raise_for_status()
soup = BeautifulSoup(response.content, "html.parser")

# Query Soup Object
found_url = {}
for a in soup.find_all("a", href=True):
    # print(a["href"])
    href = a["href"]
    if "control-area-balance-daily.csv" in href:
        found_url["daily"] = href
    elif "control-area-balance-2026.csv" in href:
        found_url["2026"] = href
if not found_url:
        raise RuntimeError("No control-area-balance CSV links found; page layout changed.")

print(found_url)

{'daily': 'https://www.swissgrid.ch/dam/jcr:0bbafc76-e5bc-4126-bd99-41c939fe10e0/control-area-balance-daily.csv', '2026': 'https://www.swissgrid.ch/dam/jcr:e329e825-6d06-4713-9551-54e301f32095/control-area-balance-2026.xlsx'}


In [ ]:
def discover_urls() -> dict[str, str]:
    resp = requests.get(PAGE_URL, timeout=30)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")
    found = {}
    for a in soup.find_all("a", href=True):
        m = FILE_RE.search(a["href"])
        if m:
            found[m.group(0)] = urljoin(PAGE_URL, a["href"])
    if not found:
        raise RuntimeError("No control-area-balance CSV links found; page layout changed?")
    return found

In [16]:
files = discover_urls()
daily_url = files["daily"]
url_2026 = files["2026"]


In [ ]:
from pathlib import Path

RAW_DIR = Path("../data/raw_data")
RAW_DIR.mkdir(parents=True, exist_ok=True)

def download_safe_file(url: str, filename: str) -> None:
    response = requests.get(url, timeout=60)
    response.raise_for_status()
    (RAW_DIR / filename).write_bytes(response.content)

download_safe_file(daily_url, "control-area-balance-daily.csv")
download_safe_file(url_2026, "control-area-balance-2026.csv")
# Also download all older files.

In [22]:
import pandas as pd
import numpy as np

df_daily = pd.read_csv("../data/raw_data/control-area-balance-daily.csv", sep=";")
df_daily.head()

,Date Time [UTC],Abgedeckte Bedarf der aFRR+,Abgedeckte Bedarf der aFRR-,Abgedeckte Bedarf der SA mFRR+,Abgedeckte Bedarf der SA mFRR-,Abgedeckte Bedarf der DA mFRR+,Abgedeckte Bedarf der DA mFRR-,NRV+ (Import),NRV- (Export),FRCE+ (Import),FRCE- (Export),Total System Imbalance,AE-Preis
0,12.10.2026 17:00,114.452302,0.000000,30,0,0,0,6.018622,-2.005756,24.479453,-18.551972,-154.392650,37.95
1,12.10.2026 17:15,81.266571,-1.127651,37,0,0,0,19.972030,-2.965511,16.641713,-28.493660,-122.293492,41.85
2,12.10.2026 17:30,3.569061,-6.586383,48,0,0,0,34.588040,-2.750270,17.034494,-16.708178,-77.146764,30.02
3,12.10.2026 17:45,2.163618,-15.087089,0,0,0,0,71.346944,-1.651111,13.439668,-14.870057,-55.341973,26.89
4,12.10.2026 18:00,0.000000,-58.552501,61,0,0,0,2.000867,-3.072104,12.570526,-17.651374,3.704588,13.96
